# Reproducing Granger and Newbold's Table 1

Goal: generate two *independent* random walks, regress one on the other, repeat 100 times, and check how often we'd wrongly conclude there's a significant relationship at the usual 5% level (|t| > ~2). The paper found this happens on ~75% of trials, not ~5%.


In [1]:
import numpy as np

np.random.seed(0)


## Step 1: simulate a single random walk

In [2]:
def simulate_random_walk(T):
    random_walk = np.random.randn(T)
    return np.cumsum(random_walk)


# Step 2: Generate 2 different random walks and regress 

In [ ]:
import statsmodels.api as sm

series_1 = simulate_random_walk(100)
series_2 = simulate_random_walk(100)

x_with_const = sm.add_constant(series_2)
model = sm.OLS(series_1,x_with_const)
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.361
Model:                            OLS   Adj. R-squared:                  0.355
Method:                 Least Squares   F-statistic:                     55.38
Date:                Wed, 16 Sep 2026   Prob (F-statistic):           3.86e-11
Time:                        08:17:57   Log-Likelihood:                -225.06
No. Observations:                 100   AIC:                             454.1
Df Residuals:                      98   BIC:                             459.3
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -4.1514      0.437     -9.497      0.0

# Step 3: Wrap as function

In [58]:
def generate_results(n):
    
    r_squared_array = []
    
    for _ in range(0,n):
        series_1 = simulate_random_walk(100)
        series_2 = simulate_random_walk(100)

        x_with_const = sm.add_constant(series_2)
        model = sm.OLS(series_1,x_with_const)
        results = model.fit()
        r_squared_array.append((results.rsquared, abs(results.tvalues[1])))
    return r_squared_array

# Step 4: Test Research findings

In [59]:
results = generate_results(100)
results_array = np.array(results)

r_squared_mean, t_test_mean = np.mean(results_array[:,0]), np.mean(results_array[:,1])
print(r_squared_mean)
print(t_test_mean)

0.24351432973919593
5.860573608408481


In [66]:
t_test_more_than_2_count = sum(results_array[:,1] > 2)
print(t_test_more_than_2_count / 100)

0.79
